In [ ]:
import os
from openai import OpenAI, AsyncOpenAI
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
import asyncio
from pydantic import BaseModel, Field
from agents import Agent, Runner, trace, function_tool, SQLiteSession, OpenAIChatCompletionsModel
load_dotenv(override=True)
from agents import GuardrailFunctionOutput, output_guardrail

In [ ]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
google_api_key = os.getenv("Gemini_API")
gemini_client = AsyncOpenAI(base_url = GEMINI_BASE_URL, api_key=google_api_key)
gemini_model = OpenAIChatCompletionsModel(model="gemini-3.5-flash-lite", openai_client=gemini_client)

In [ ]:
class EmailReview(BaseModel):
    is_professional: bool = Field(description="whether the email is professional or appropriate")
    number_of_sentences: int = Field(description="The number of sentences in body of the email not including greeting signature")
    contains_placeholders: bool = Field(description="Whether email contains placeholders for personalization")

In [ ]:
EmailReview.model_json_schema()

In [ ]:
email = """ Hi [first name], I am hitting you up to see if you would like to buy our product. you will miss out if you do not buy it.
Laters.
Rohit"""

# just set output_type as the pydantic
checker = Agent(name="Checker", instructions="You review sales emails", model=gemini_model, output_type=EmailReview)
result = await Runner.run(checker, email)

In [ ]:
review = result.final_output

In [ ]:
review

## Guardrails!!

In [ ]:
@output_guardrail
async def email_guardrail(ctx, agent, message):
    result = await Runner.run(checker, message, context=ctx.context)
    review = result.final_output
    is_problem = review.contains_placeholders or not review.is_professional
    return GuardrailFunctionOutput(output_info={"review": review}, tripwire_triggered=is_problem)

In [ ]:
cowboy_instructions = "You are here to draft an email but speak like a cowboy"

sales_agent_cowboy = Agent(name="Cowboy", instructions=cowboy_instructions, model=gemini_model, output_guardrails=[email_guardrail])

In [ ]:
result =  await Runner.run(sales_agent_cowboy, "Write a cold sales email")

## Simplest guardrails using checker pydantic Runner.run calls

In [ ]:
simple_cowboy = Agent(name="Simple cowboy", instructions=cowboy_instructions, model=gemini_model)
result = await Runner.run(simple_cowboy, "Write a cold email")
email = result.final_output

In [ ]:
error_checker = await Runner.run(checker, email)
review = error_checker.final_output
if not review.is_professional or review.contains_placeholders:
    print("Email is not professional try again")
else:
    print("All Ok")